In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, random_split

import torchvision
import torchvision.transforms as transforms


In [6]:
# Load MNIST (downloads to ./data on first run) and merge the default splits
# so we can carve out our own 80/20 train/test split.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_part = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_part = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

full_dataset = ConcatDataset([train_part, test_part])

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42)
)

print(f"Total samples: {len(full_dataset)}")
print(f"Train samples: {len(train_dataset)} ({len(train_dataset)/len(full_dataset):.0%})")
print(f"Test samples: {len(test_dataset)} ({len(test_dataset)/len(full_dataset):.0%})")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Total samples: 70000
Train samples: 56000 (80%)
Test samples: 14000 (20%)


In [7]:
class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 64)
        self.fc5 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten (batch, 1, 28, 28) -> (batch, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = self.fc5(x)  # raw logits, no softmax (CrossEntropyLoss applies it)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = DigitNet().to(device)
print(model)


DigitNet(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (fc4): Linear(in_features=64, out_features=64, bias=True)
  (fc5): Linear(in_features=64, out_features=10, bias=True)
)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{num_epochs} - loss: {epoch_loss:.4f} - accuracy: {epoch_acc:.4f}")


Epoch 1/5 - loss: 0.3160 - accuracy: 0.8981
Epoch 2/5 - loss: 0.1204 - accuracy: 0.9620
Epoch 3/5 - loss: 0.0872 - accuracy: 0.9729
Epoch 4/5 - loss: 0.0673 - accuracy: 0.9785
Epoch 5/5 - loss: 0.0540 - accuracy: 0.9834


In [9]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total
print(f"Test accuracy: {test_acc:.4f}")


Test accuracy: 0.9719
